# Arquitecturas convolucionales preentrenadas

Este notebook compara ideas centrales de AlexNet, VGG, GoogLeNet y ResNet, y muestra cómo usar correctamente pesos preentrenados en ImageNet.

**Conceptos:** profundidad, número de parámetros, bloques Inception, conexiones residuales, preprocesamiento e inferencia top-k.

**Resultado esperado:** distintas arquitecturas pueden producir predicciones similares, pero difieren mucho en tamaño y organización interna.


## 1. Preparación

Usaremos la API moderna de pesos de TorchVision. Cada objeto de pesos incluye el preprocesamiento y los nombres de las clases usados durante el entrenamiento; esto evita mantener etiquetas externas o normalizaciones copiadas manualmente.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import torch
import torchvision
from PIL import Image
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("Dispositivo:", device)


Descargamos una imagen de ejemplo solo si no existe. También se puede reemplazar sample_image_path por una fotografía propia en formato RGB.

**Resultado esperado:** una fotografía de dos perros. El modelo devolverá clases de ImageNet, que no necesariamente coinciden con categorías cotidianas más generales.


In [ ]:
sample_image_path = Path("dog.jpg")
if not sample_image_path.exists():
    urlretrieve(
        "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg",
        sample_image_path,
    )

sample_image = Image.open(sample_image_path).convert("RGB")
plt.figure(figsize=(8, 6))
plt.imshow(sample_image)
plt.title("Imagen de entrada")
plt.axis("off")
plt.show()


## 2. ¿Cuánto cambia el tamaño de cada arquitectura?

El número de parámetros es una aproximación sencilla al tamaño del modelo. No mide por sí solo velocidad ni calidad: también importan las operaciones y el hardware.

- **AlexNet:** popularizó CNN profundas en ImageNet.
- **VGG11:** repite convoluciones pequeñas; es simple, pero tiene muchas capas densas.
- **GoogLeNet:** combina filtros de varios tamaños mediante bloques Inception.
- **ResNet18:** usa conexiones residuales que facilitan el flujo del gradiente.

**Resultado esperado:** VGG11 tendrá muchos más parámetros que ResNet18 y GoogLeNet.


In [ ]:
architecture_builders = {
    "AlexNet": models.alexnet,
    "VGG11": models.vgg11,
    "GoogLeNet": models.googlenet,
    "ResNet18": models.resnet18,
}

parameter_counts = {}
for architecture_name, build_model in architecture_builders.items():
    if architecture_name == "GoogLeNet":
        model_without_weights = build_model(
            weights=None,
            init_weights=False,
        )
    else:
        model_without_weights = build_model(weights=None)
    trainable_parameters = sum(
        parameter.numel()
        for parameter in model_without_weights.parameters()
    )
    parameter_counts[architecture_name] = trainable_parameters
    print(
        f"{architecture_name:10s}: "
        f"{trainable_parameters / 1_000_000:6.2f} millones"
    )


## 3. Inferencia con ResNet18

La función siguiente recibe explícitamente modelo, pesos e imagen. El preprocesamiento redimensiona, recorta, convierte a tensor y normaliza con los valores usados en ImageNet.

Los logits son puntajes sin normalizar. softmax los transforma en probabilidades que suman uno.

**Resultado esperado:** una lista top-5 ordenada por probabilidad. En una imagen con más de un objeto, una red de clasificación describe la imagen completa y no entrega cajas.


In [ ]:
def predict_top_k(model, weights, image, k=5):
    # Aplica exactamente el preprocesamiento asociado a estos pesos.
    input_tensor = weights.transforms()(image).unsqueeze(0).to(device)

    model = model.to(device)
    model.eval()
    with torch.inference_mode():
        logits = model(input_tensor)
        probabilities = torch.softmax(logits, dim=1)[0]

    top_probabilities, top_indices = probabilities.topk(k)
    class_names = weights.meta["categories"]

    predictions = []
    for probability, class_index in zip(
        top_probabilities.cpu(),
        top_indices.cpu(),
    ):
        predictions.append(
            {
                "clase": class_names[class_index.item()],
                "probabilidad": probability.item(),
            }
        )
    return predictions


resnet_weights = models.ResNet18_Weights.DEFAULT
resnet18 = models.resnet18(weights=resnet_weights)
resnet_predictions = predict_top_k(
    resnet18,
    resnet_weights,
    sample_image,
)

for rank, prediction in enumerate(resnet_predictions, start=1):
    print(
        f"{rank}. {prediction['clase']}: "
        f"{100 * prediction['probabilidad']:.2f} %"
    )


## 4. Comparación de arquitecturas

Cargamos un modelo a la vez para no acumular innecesariamente memoria. Cada arquitectura usa su propio objeto de pesos y, por lo tanto, su preprocesamiento correcto.

**Resultado esperado:** las etiquetas o el orden pueden cambiar. No se debe concluir que una arquitectura es mejor usando una sola imagen.


In [ ]:
pretrained_configurations = [
    (
        "AlexNet",
        models.alexnet,
        models.AlexNet_Weights.DEFAULT,
    ),
    (
        "GoogLeNet",
        models.googlenet,
        models.GoogLeNet_Weights.DEFAULT,
    ),
    (
        "ResNet18",
        models.resnet18,
        models.ResNet18_Weights.DEFAULT,
    ),
]

comparison = {}
for architecture_name, build_model, weights in pretrained_configurations:
    current_model = build_model(weights=weights)
    comparison[architecture_name] = predict_top_k(
        current_model,
        weights,
        sample_image,
        k=3,
    )
    del current_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

for architecture_name, predictions in comparison.items():
    print(f"\n{architecture_name}")
    for prediction in predictions:
        print(
            f"  {prediction['clase']:25s} "
            f"{100 * prediction['probabilidad']:6.2f} %"
        )


## 5. Inspección de ResNet18

Imprimimos solo los bloques principales. layer1 a layer4 contienen bloques residuales; avgpool resume cada mapa espacial y fc produce 1000 logits de ImageNet.


In [ ]:
for block_name, block in resnet18.named_children():
    output_description = block.__class__.__name__
    print(f"{block_name:10s} → {output_description}")

print(
    "\nEntradas de la última capa:",
    resnet18.fc.in_features,
)
print(
    "Clases de salida:",
    resnet18.fc.out_features,
)


## 6. Ejercicios

**Ejercicio 1 — Cambio de dominio.** Use una imagen que no pertenezca claramente a ImageNet, por ejemplo un diagrama o una radiografía. Observe el top-5.

**Resultado esperado:** softmax siempre reparte probabilidad entre las clases disponibles, incluso si ninguna describe bien la entrada.

**Ejercicio 2 — Parámetros y memoria.** Calcule el tamaño aproximado de los parámetros suponiendo 4 bytes por float32. Compare VGG11 y ResNet18.

**Resultado esperado:** tamaño aproximado = número de parámetros × 4 bytes; no incluye activaciones ni estado del optimizador.

**Ejercicio 3 — Clasificación versus detección.** Explique por qué las predicciones anteriores no indican dónde está cada perro.

**Resultado esperado:** el clasificador produce un vector por imagen; un detector debe producir además localización y posiblemente varias instancias.
